In [7]:
# ==== Minimal SU(3) Metropolis Yang–Mills + Q (NumPy/SciPy) ====
# Paste this whole cell into Colab and run.
# You can change L, n_sweeps, etc. at the bottom if you want.

import numpy as np
from numpy.random import rand, randn, randint
import scipy.linalg as la
import math
import itertools
import time

# -----------------------------
# Lattice / parameters
# -----------------------------

NDIM = 4  # 4D lattice

# You CAN change these after first run, but you don't have to.
L = 4                 # lattice size in each dimension
beta = 5.8            # gauge coupling parameter
n_therm_sweeps = 20   # number of thermalization sweeps
n_meas_sweeps = 50    # number of sweeps where we record W,Q
meas_interval = 1     # record once per sweep

# -----------------------------
# Index helper
# -----------------------------

def add_dir(x, mu, sign, L):
    """Add sign * e_mu to coordinate x=(i,j,k,l) with periodic BC."""
    i, j, k, l = x
    if mu == 0:
        i = (i + sign) % L
    elif mu == 1:
        j = (j + sign) % L
    elif mu == 2:
        k = (k + sign) % L
    else:
        l = (l + sign) % L
    return (i, j, k, l)


# -----------------------------
# SU(3) utilities
# -----------------------------

def random_su3_generator(scale=0.3):
    """
    Random su(3) Lie algebra element:
    anti-Hermitian, traceless 3x3 complex, with norm ~ scale.
    """
    A = randn(3,3) + 1j*randn(3,3)
    H = 0.5*(A - A.conj().T)   # anti-Hermitian
    tr = np.trace(H)/3.0
    H = H - tr*np.eye(3, dtype=complex)
    # rescale
    norm = np.sqrt(np.real(np.trace(H.conj().T @ H)))
    if norm > 1e-12:
        H = (scale/norm)*H
    return H


def su3_unit():
    """Identity SU(3)."""
    return np.eye(3, dtype=complex)


# -----------------------------
# Plaquettes and action
# -----------------------------

def plaquette(U, x, mu, nu, L):
    """
    Single plaquette at site x in plane (mu,nu):
      U_mu(x) U_nu(x+mu) U_mu^\dagger(x+nu) U_nu^\dagger(x).
    U shape: (L,L,L,L,NDIM,3,3)
    """
    i,j,k,l = x
    x_mu = add_dir(x, mu, +1, L)
    x_nu = add_dir(x, nu, +1, L)

    U_mu_x     = U[i, j, k, l, mu]
    U_nu_xmu   = U[x_mu[0], x_mu[1], x_mu[2], x_mu[3], nu]
    U_mu_dag_xnu = U[x_nu[0], x_nu[1], x_nu[2], x_nu[3], mu].conj().T
    U_nu_dag_x   = U[i, j, k, l, nu].conj().T

    return U_mu_x @ U_nu_xmu @ U_mu_dag_xnu @ U_nu_dag_x


def wilson_action(U, beta):
    """
    Wilson action:
      S = beta/3 * sum_p Re Tr(1 - U_p)
    """
    L = U.shape[0]
    S = 0.0
    for mu in range(NDIM):
        for nu in range(mu+1, NDIM):
            for i in range(L):
                for j in range(L):
                    for k in range(L):
                        for l in range(L):
                            Up = plaquette(U, (i,j,k,l), mu, nu, L)
                            S += beta/3.0 * (3.0 - np.real(np.trace(Up)))
    return S


def avg_plaquette(U):
    """
    Volume-averaged plaquette Re Tr(U_p)/3.
    """
    L = U.shape[0]
    total = 0.0
    count = 0
    for mu in range(NDIM):
        for nu in range(mu + 1, NDIM):
            for i in range(L):
                for j in range(L):
                    for k in range(L):
                        for l in range(L):
                            Up = plaquette(U, (i,j,k,l), mu, nu, L)
                            total += np.real(np.trace(Up))/3.0
                            count += 1
    return total / count


# -----------------------------
# Local Metropolis update
# -----------------------------

def local_metropolis_update(U, beta, eps=0.3):
    """
    One Metropolis sweep over the whole lattice:
    - For each link (x,mu), propose U' = exp(eps*H) U, H in su(3),
      accept with prob min(1, exp(-ΔS)).
    WARNING: this recomputes the *full* action for simplicity.
    This is slow but robust and easy to reason about.
    """
    L = U.shape[0]
    S_old = wilson_action(U, beta)

    for i in range(L):
        for j in range(L):
            for k in range(L):
                for l in range(L):
                    for mu in range(NDIM):
                        x = (i,j,k,l)
                        # save old link
                        U_old = U[i,j,k,l,mu].copy()

                        # propose new link
                        H = random_su3_generator(scale=eps)
                        U_prop = la.expm(H) @ U_old

                        # update field
                        U[i,j,k,l,mu] = U_prop
                        # compute new action
                        S_new = wilson_action(U, beta)
                        dS = S_new - S_old
                        if dS <= 0.0 or rand() < math.exp(-dS):
                            # accept
                            S_old = S_new
                        else:
                            # reject
                            U[i,j,k,l,mu] = U_old
    return U, S_old


# -----------------------------
# Clover field strength and Q
# -----------------------------

def clover_F(U, x, mu, nu):
    """
    Clover F_{mu,nu}(x) built from 4 plaquettes around x:
      x, x-mu, x-nu, x-mu-nu
    F ~ (1/(8i)) [sum plaquettes - h.c.].
    """
    L = U.shape[0]
    xs = [
        x,
        add_dir(x, mu, -1, L),
        add_dir(x, nu, -1, L),
        add_dir(add_dir(x, mu, -1, L), nu, -1, L),
    ]
    sum_p = np.zeros((3,3), dtype=complex)
    for y in xs:
        sum_p += plaquette(U, y, mu, nu, L)
    F = (sum_p - sum_p.conj().T) / (8j)
    return F


def levi_civita_4():
    """
    4D Levi-Civita epsilon_{mu nu rho sigma}
    """
    eps = np.zeros((4,4,4,4), dtype=int)
    for perm in itertools.permutations(range(4)):
        mu,nu,rho,sigma = perm
        mat = np.eye(4)[list(perm)]
        sg = round(np.linalg.det(mat))
        eps[mu,nu,rho,sigma] = int(sg)
    return eps

EPS4 = levi_civita_4()


def topo_charge(U):
    """
    Very crude topological charge estimator using clover F_{mu,nu}.
    Q = 1/(32 pi^2) sum_x eps_{mu nu rho sigma} Tr(F_{mu nu} F_{rho sigma})
    Note: no gradient flow here; this is noisy but OK for a smoke test.
    """
    L = U.shape[0]
    # Precompute F_{mu,nu}(x)
    Fmn = {}
    for i in range(L):
        for j in range(L):
            for k in range(L):
                for l in range(L):
                    x = (i,j,k,l)
                    for mu in range(NDIM):
                        for nu in range(mu+1, NDIM):
                            Fmn[(x,mu,nu)] = clover_F(U, x, mu, nu)

    Q = 0.0
    for i in range(L):
        for j in range(L):
            for k in range(L):
                for l in range(L):
                    x = (i,j,k,l)
                    for mu in range(NDIM):
                        for nu in range(mu+1, NDIM):
                            F1 = Fmn[(x,mu,nu)]
                            for rho in range(NDIM):
                                for sigma in range(rho+1, NDIM):

                                    F2 = Fmn[(x,rho,sigma)]
                                    e = EPS4[mu,nu,rho,sigma]
                                    if e == 0:
                                        continue
                                    Q += e * np.trace(F1 @ F2).real
    Q /= (32 * math.pi**2)
    return Q


# -----------------------------
# Autocorrelation utilities
# -----------------------------

def autocorr(x):
    """
    Naive autocorrelation function C(t) normalized by variance.
    """
    x = np.asarray(x, dtype=float)
    n = x.shape[0]
    x_mean = x.mean()
    y = x - x_mean
    var = np.mean(y*y)
    C = np.zeros(n)
    for t in range(n):
        C[t] = np.mean(y[:n-t]*y[t:])
    return C / var


def integrated_autocorr_time(C, window=None):
    """
    Integrated autocorrelation time tau_int from C(t).
    """
    C = np.asarray(C, dtype=float)
    if window is None:
        window = len(C)//10 if len(C) > 10 else len(C)
    return 0.5 + C[1:window].sum()


# -----------------------------
# Main driver
# -----------------------------

def run_simulation():
    """
    Full run: initialize, thermalize, measure W and Q, compute tau_int.
    """
    # Initial configuration: all links = identity
    U = np.zeros((L,L,L,L,NDIM,3,3), dtype=complex)
    for i in range(L):
        for j in range(L):
            for k in range(L):
                for l in range(L):
                    for mu in range(NDIM):
                        U[i,j,k,l,mu] = su3_unit()

    print(f"Starting SU(3) Metropolis run: L={L}, beta={beta}")
    print("Thermalizing...")
    S = wilson_action(U, beta)
    for sweep in range(n_therm_sweeps):
        U, S = local_metropolis_update(U, beta, eps=0.3)
        print(f"  therm sweep {sweep+1}/{n_therm_sweeps}, S={S:.4f}")

    print("\nMeasuring...")
    Ws = []
    Qs = []
    sweep_count = 0
    for sweep in range(n_meas_sweeps):
        U, S = local_metropolis_update(U, beta, eps=0.3)
        sweep_count += 1
        if sweep_count % meas_interval == 0:
            W = avg_plaquette(U)
            Q = topo_charge(U)
            Ws.append(W)
            Qs.append(Q)
            print(f"  meas sweep {sweep_count}: W={W:.4f}, Q={Q:.3f}, S={S:.4f}")

    Ws = np.array(Ws)
    Qs = np.array(Qs)

    print("\nComputing autocorrelations...")
    C_W = autocorr(Ws)
    C_Q = autocorr(Qs)

    tau_W = integrated_autocorr_time(C_W)
    tau_Q = integrated_autocorr_time(C_Q)

    print(f"\nResults (L={L}, beta={beta}):")
    print(f"  # measurements: {len(Ws)}")
    print(f"  mean(W) = {Ws.mean():.4f}, tau_int(W) ~ {tau_W:.2f}")
    print(f"  mean(Q) = {Qs.mean():.4f}, tau_int(Q) ~ {tau_Q:.2f}")

    return U, Ws, Qs, tau_W, tau_Q


# Actually run it
start_time = time.time()
U_final, Ws, Qs, tau_W, tau_Q = run_simulation()
end_time = time.time()
print(f"\nTotal wall-clock time: {end_time - start_time:.1f} s")


<>:75: SyntaxWarning: invalid escape sequence '\d'
<>:75: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipython-input-3260281283.py:75: SyntaxWarning: invalid escape sequence '\d'
  U_mu(x) U_nu(x+mu) U_mu^\dagger(x+nu) U_nu^\dagger(x).


Starting SU(3) Metropolis run: L=4, beta=5.8
Thermalizing...
  therm sweep 1/20, S=318.1576
  therm sweep 2/20, S=563.1713
  therm sweep 3/20, S=768.1341
  therm sweep 4/20, S=972.7397
  therm sweep 5/20, S=1130.2176
  therm sweep 6/20, S=1270.9186
  therm sweep 7/20, S=1388.9639
  therm sweep 8/20, S=1493.2843
  therm sweep 9/20, S=1615.7221
  therm sweep 10/20, S=1694.3302
  therm sweep 11/20, S=1772.9031
  therm sweep 12/20, S=1866.5784
  therm sweep 13/20, S=1941.0634
  therm sweep 14/20, S=2014.0872
  therm sweep 15/20, S=2097.3501
  therm sweep 16/20, S=2164.8545
  therm sweep 17/20, S=2236.3829
  therm sweep 18/20, S=2293.2761
  therm sweep 19/20, S=2334.4876
  therm sweep 20/20, S=2438.7437

Measuring...
  meas sweep 1: W=0.7220, Q=0.021, S=2477.0912
  meas sweep 2: W=0.7126, Q=0.023, S=2560.7003
  meas sweep 3: W=0.7066, Q=0.005, S=2613.6821
  meas sweep 4: W=0.7012, Q=-0.002, S=2662.1214
  meas sweep 5: W=0.6970, Q=-0.014, S=2699.6456
  meas sweep 6: W=0.6919, Q=-0.021, S=274